# 16 · Fabric IQ data agents

## Goal

Ground spend questions in a real Fabric semantic model instead of a mocked
Dataverse table, with the compliance-boundary consequence understood and
accepted (or not) before a single query runs — not discovered afterward.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
settings = load_settings()
settings.require("FABRIC_WORKSPACE_ID")


In [ ]:
from csx.checkpoint import checkpoint
checkpoint(
    name="Fabric capacity is F2 or higher, and cross-geo processing/storing tenant setting is understood",
    probe=lambda: input("Confirmed F2+ capacity AND read the compliance-boundary note below? (y/n): ") == "y",
    remediation="Check Fabric capacity SKU in the Fabric admin portal. Read the Concept section below before answering yes.",
)


## Concept

**Lead with this, not the demo — finding #11.** Fabric data agent responses
in Copilot Studio may leave Fabric's compliance boundary and geo. This
requires F2+ Fabric capacity and the tenant's *cross-geo processing/storing*
setting to be explicitly on. If your supplier spend data has residency
requirements, that is a real decision to make with your compliance team
before this notebook's first query runs — not a checkbox to click past.

Once that's settled: this is standard semantic-model grounding with
natural-language-to-query guardrails — the agent doesn't get raw SQL/DAX
access, it gets a governed query surface the semantic model defines.
`fabric-02-nl-to-query-guardrail` in the golden set exists to prove a
destructive-sounding request ("drop the table") is refused, not attempted.


## Build


### Attach the Fabric semantic model as knowledge


In [ ]:
import yaml
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")

source = {
    "id": "fabric-spend-semantic-model",
    "type": "fabric_data_agent",
    "displayName": "Supplier spend (Fabric semantic model)",
    "description": "Governed NL-to-query access to the supplier spend semantic model. Read-only; no schema-modifying operations are exposed.",
    "workspaceId": settings.get("FABRIC_WORKSPACE_ID"),
    "semanticModel": "SupplierSpendModel",
}
(workspace / "knowledge" / "fabric-spend-semantic-model.yaml").write_text(yaml.dump(source, sort_keys=False))

from csx.pac import copilot_push
import subprocess
copilot_push(workspace)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
suite = run_suite(client, cases=load_golden(tags=["fabric"]) + load_golden(tags=["core"]), credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("16", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="Fabric-grounded queries + NL-to-query guardrail check")


## Teardown


In [ ]:
print("No teardown — Fabric semantic model source persists for 17's ontology work.")
